In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url=os.environ.get("GITHUB_BASE_URL"),
    api_key=os.environ.get("GITHUB_TOKEN")
)

# 1. El contexto enriquecido (RAG simulado)
contexto_red = """
INFORMACIÓN DE TARIFAS (General):
- Horario Punta (07:00-08:59 y 18:00-19:59): $840
- Horario Valle (09:00-17:59 y 20:00-20:44): $760
- Horario Bajo (06:00-06:59 y 20:45-23:00): $680
- Tarifa Estudiante (TNE) y Adulto Mayor: $240 en todo horario.

HORARIOS DE FUNCIONAMIENTO DE LA RED:
- Lunes a Viernes: 06:00 a 23:00 hrs.
- Sábados: 06:30 a 23:00 hrs.
- Domingos y Festivos: 08:00 a 23:00 hrs.

ESTRUCTURA DE LÍNEAS Y COMBINACIONES CLAVE:
- Línea 1 (Roja): Terminales San Pablo / Los Dominicos. Combina con L2 (Los Héroes), L3 (U. de Chile), L4 (Tobalaba), L5 (Baquedano / San Pablo) y L6 (Los Leones).
- Línea 4 (Azul): Terminales Tobalaba / Plaza Puente Alto. Combina con L1 (Tobalaba), L3 (Plaza Egaña) y L5 (Vicente Valdés).
- Línea 5 (Verde): Terminales Plaza de Maipú / Vicente Valdés. Combina con L1, L2, L3, L4 y L6.
"""

horario_actual = "Día hábil, 18:30 hrs (Horario Punta)"
alerta_operativa = "Línea 1 presenta retrasos por asistencia a pasajero en estación Baquedano. Resto de la red operativa."

# 2. El Prompt Optimizado (¡Este bloque te faltaba!)
prompt_sistema = f"""
Actúa como el asistente virtual oficial de servicio al cliente de Metro de Santiago. Tu tarea es ayudar a los pasajeros a planificar su viaje utilizando estrictamente la información proporcionada en el contexto.

Contexto de la red: {contexto_red}
Horario actual: {horario_actual}
Alertas operativas: {alerta_operativa}

Reglas:
1. Indica claramente la línea inicial, las estaciones donde se debe hacer combinación y la estación final.
2. Informa la tarifa exacta del viaje basándote en el horario actual.
3. Si existe alguna alerta operativa en la ruta solicitada, debes advertirlo inmediatamente.
4. Si el usuario hace una consulta que no tiene relación con el transporte, responde únicamente: 'Soy el asistente de Metro de Santiago y solo puedo ayudarte con información de la red'.
"""

# 3. La ejecución
try:
    print("Consultando al asistente de Metro...\n")
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": "Hola, necesito ir desde Plaza Puente Alto (L4) hasta Pedro de Valdivia (L1). ¿Cuánto me cuesta ahora y dónde combino?"}
        ],
        temperature=0.7, 
        max_tokens=600
    )
    
    print("=== RESPUESTA DEL ASISTENTE ===")
    print(response.choices[0].message.content)
    
    print("\n=== CONSUMO DE TOKENS ===")
    print(f"Tokens totales usados: {response.usage.total_tokens}")

except Exception as e:
    print(f"Error de conexión: {e}")

Consultando al asistente de Metro...

=== RESPUESTA DEL ASISTENTE ===
Para ir desde **Plaza Puente Alto (L4)** hasta **Pedro de Valdivia (L1)**, sigue estos pasos:

1. Toma la **Línea 4 (Azul)** en dirección a **Tobalaba**.
2. Combina en la estación **Tobalaba** con la **Línea 1 (Roja)**.
3. Toma la **Línea 1 (Roja)** en dirección a **San Pablo** y bájate en la estación **Pedro de Valdivia**.

### Tarifa:
El horario actual es **Punta (18:30 hrs)**, por lo que la tarifa es de **$840**.

### Alerta operativa:
Debes tener en cuenta que la **Línea 1** presenta retrasos debido a asistencia a un pasajero en la estación Baquedano. Te sugiero considerar tiempos adicionales para tu viaje.

=== CONSUMO DE TOKENS ===
Tokens totales usados: 759
